# 04. YOLO weight로 Label Studio prediction 생성

이 notebook은 이미 Label Studio project에 import된 image task를 대상으로 Ultralytics YOLO 계열 weight를 실행하고, 결과 bbox를 Label Studio의 `prediction`으로 업로드합니다.

안전 기본값은 `DRY_RUN=True`입니다. 이 상태에서는 모델을 실행하거나 prediction을 업로드하지 않고, project/task/image 경로와 class 설정만 확인합니다.

핵심 용어:
- `CONF`: 모델이 bbox 후보를 만들 때 쓰는 기본 confidence threshold입니다.
- `IOU`: Ultralytics 내부 NMS에 들어가는 기본 IoU threshold입니다.
- `CLASS_THRESH`: class별 confidence threshold입니다. 예를 들어 `helmet`은 0.4, `worker`는 0.25처럼 따로 조정할 수 있습니다.
- `CLASS_IOU`: 모델 결과에 대해 class별 NMS를 한 번 더 적용할 때 쓰는 IoU threshold입니다. 서로 비슷한 class가 겹쳐 나오는 경우를 줄일 때 사용합니다.
- `META_TAG`, `IMPORT_ID`, `PRED_MODEL`: 나중에 merge/delete/export에서 결과를 구분하기 위한 이름표입니다.


In [ ]:
from pathlib import Path
import sys

# notebook을 repo 어느 위치에서 열어도 src package를 찾기 위한 bootstrap입니다.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from labelstudio_bbox_tools.config import settings_from_env

settings = settings_from_env(REPO_ROOT / ".env")
print("repo:", REPO_ROOT)
print("Label Studio URL:", settings.url)
print("doc_root:", settings.doc_root)


In [ ]:
from pathlib import Path
from labelstudio_bbox_tools.pseudo_label.yolo import auto_label_yolo, load_yolo_class_names

# ===== 사용자 설정 =====
PROJECT_ID = 0

# weight 파일 경로입니다. 예: Path("/path/to/best.pt")
WEIGHTS_YOLO = Path("/path/to/yolo11_best.pt")

# 권장: YOLO dataset yaml의 names 필드에서 class 순서를 자동으로 읽습니다.
# yaml을 쓰면 MANUAL_CLASSES보다 우선됩니다.
CLASS_YAML = Path("/path/to/data.yaml")

# fallback: yaml이 없을 때만 수기로 class 순서를 적습니다.
MANUAL_CLASSES = []

IMG_SIZE = 640
BATCH_SIZE = 1  # 기존 경험상 1이 가장 안정적입니다.
CONF = 0.30
IOU = 0.60

# class별 confidence threshold입니다. 없는 class는 default를 사용합니다.
CLASS_THRESH = {
    "default": CONF,
}

# class별 2차 NMS IoU threshold입니다. 없는 class는 default 또는 IOU를 사용합니다.
# 값이 낮을수록 겹치는 bbox를 더 강하게 제거합니다.
CLASS_IOU = {
    "default": IOU,
}

META_TAG = "my_yolo_pseudo_label_run"
IMPORT_ID = "my_import_id"
PRED_MODEL = META_TAG

SAVE_LOCAL = False
LOCAL_OUT_DIR = REPO_ROOT / "outputs" / "pseudo_label_preview"
SAVE_FORMAT = "json"

MAX_TASKS = 20      # 처음 테스트할 때는 작은 값으로 확인하세요.
DRY_RUN = True      # False로 바꾸면 실제 prediction을 업로드합니다.


In [ ]:
# class 목록 확인 cell입니다.
# CLASS_YAML 경로가 아직 placeholder라면 에러가 납니다. 그 경우 CLASS_YAML=None으로 두고 MANUAL_CLASSES를 채우세요.
class_yaml = CLASS_YAML if CLASS_YAML and CLASS_YAML.exists() else None
classes = load_yolo_class_names(class_yaml=class_yaml, manual_classes=MANUAL_CLASSES)
print("class count:", len(classes))
print(classes)


In [ ]:
summary = auto_label_yolo(
    ls_url=settings.url,
    api_key=settings.api_key,
    project_id=PROJECT_ID,
    model_weights=WEIGHTS_YOLO,
    doc_root=settings.doc_root,
    class_yaml=class_yaml,
    classes=MANUAL_CLASSES,
    device="cuda",
    imgsz=IMG_SIZE,
    conf=CONF,
    iou=IOU,
    batch_size=BATCH_SIZE,
    class_conf_map=CLASS_THRESH,
    class_iou_map=CLASS_IOU,
    save_local=SAVE_LOCAL,
    local_out_dir=LOCAL_OUT_DIR,
    save_format=SAVE_FORMAT,
    pred_model=PRED_MODEL,
    import_id=IMPORT_ID,
    meta_tag=META_TAG,
    dry_run=DRY_RUN,
    max_tasks=MAX_TASKS,
)
summary.as_dict()
